# 👁️ Build a Vision AI App with Python & Florence-2

**Author:** Johnny Wilson Dougherty  
**GitHub:** [https://github.com/JohnnyWilson16](https://github.com/JohnnyWilson16)  
**Email:** johnnydougherty09@gmail.com  
**License:** MIT License  

---

## Overview
In this comprehensive walkthrough, we build an end-to-end Vision AI application powered by Microsoft's open-source **Florence-2** Vision-Language Model (VLM).

### How Florence-2 Works
Traditional computer vision pipelines required specialized separate architectures for object detection, text recognition (OCR), and image captioning. **Florence-2** unifies these tasks under a sequence-to-sequence paradigm:
1. **DaViT Vision Encoder**: Translates raw pixel patches into dense multi-scale visual tokens.
2. **Text Encoder**: Embeds task prompts (such as `<OD>`, `<OCR>`, `<CAPTION>`).
3. **Transformer Decoder**: Generates text representations and spatial coordinates dynamically.
4. **Pre-training Scale**: Trained on the massive **FLD-5B** dataset containing 5.4 billion annotations across 126 million images.

## Step 1: Install & Import Dependencies

In [ ]:
# Run installation if running in a fresh environment
# !pip install -q transformers torch torchvision pillow requests accelerate timm einops matplotlib

In [ ]:
import io
import time
import requests
import torch
from PIL import Image, ImageDraw, ImageOps
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from transformers import AutoProcessor, AutoModelForCausalLM

# Hardware accelerator check
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float16 if device in ["cuda", "mps"] else torch.float32

print(f"[✓] Active Device: {device.upper()}")
print(f"[✓] Tensor Precision: {dtype}")

## Step 2: Initialize the Model and AutoProcessor

We load `florence-community/Florence-2-base` with automatic memory allocation.

In [ ]:
model_id = "florence-community/Florence-2-base"

print(f"[*] Loading model and processor for {model_id}...")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

if device == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto",
        trust_remote_code=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        trust_remote_code=True
    ).to(device)

model.eval()
print("[✓] Model and processor successfully loaded!")

## Step 3: Load and Prepare the Image (RGBA-Safe Preprocessing)

A common pitfall in vision pipelines is passing RGBA transparent images to models expecting 3-channel RGB tensors. We implement a robust converter.

In [ ]:
def prepare_rgb_image(image: Image.Image) -> Image.Image:
    """Convert any image mode to RGB and handle transparency."""
    try:
        image = ImageOps.exif_transpose(image)
    except Exception:
        pass
    if image.mode == "RGB":
        return image
    if image.mode in ("RGBA", "LA") or (image.mode == "P" and "transparency" in image.info):
        alpha = image.convert("RGBA")
        bg = Image.new("RGB", alpha.size, (255, 255, 255))
        bg.paste(alpha, mask=alpha.split()[3])
        return bg
    return image.convert("RGB")

# Load sample image
sample_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(sample_url, headers=headers, stream=True)
raw_img = Image.open(io.BytesIO(response.content))
image = prepare_rgb_image(raw_img)

print(f"[✓] Loaded Image: {image.size[0]}x{image.size[1]} px, Mode: {image.mode}")
plt.figure(figsize=(8, 5))
plt.imshow(image)
plt.title("Input Image")
plt.axis("off")
plt.show()

## Step 4: Define the Task and Process Inputs

We select `<OD>` for Object Detection and prepare input tensors.

In [ ]:
task_prompt = "<OD>"

inputs = processor(
    text=task_prompt,
    images=image,
    return_tensors="pt"
)

# Send tensors to active compute device
inputs = {k: (v.to(device, dtype=dtype) if k == "pixel_values" and dtype != torch.float32 else v.to(device)) 
          for k, v in inputs.items()}
print("[✓] Processed inputs ready for generation.")

## Step 5: Generate Tokens and Decode Structured Predictions

We perform beam search (`num_beams=3`) and post-process token IDs into normalized pixel coordinates.

In [ ]:
start_time = time.time()

with torch.inference_mode():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3
    )

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]

parsed_answer = processor.post_process_generation(
    generated_text,
    task=task_prompt,
    image_size=(image.width, image.height)
)

elapsed = (time.time() - start_time) * 1000.0
print(f"[✓] Inference completed in {elapsed:.2f} ms\n")
print("Structured Output:")
print(parsed_answer)

## Step 6: Visual Overlay (Bounding Boxes & Badges)

Let's render high-contrast bounding boxes with category badges.

In [ ]:
colors = ["#FF3B30", "#34C759", "#007AFF", "#AF52DE", "#FF9500", "#00C7BE"]

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(image)
ax.axis("off")

detection_data = parsed_answer.get("<OD>", {})
bboxes = detection_data.get("bboxes", [])
labels = detection_data.get("labels", [])

color_map = {}
for idx, box in enumerate(bboxes):
    xmin, ymin, xmax, ymax = box
    label = labels[idx] if idx < len(labels) else "Object"
    
    if label not in color_map:
        color_map[label] = colors[len(color_map) % len(colors)]
    color = color_map[label]
    
    # Add rectangle patch
    rect = patches.Rectangle(
        (xmin, ymin),
        xmax - xmin,
        ymax - ymin,
        linewidth=2.5,
        edgecolor=color,
        facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(
        xmin,
        max(0, ymin - 5),
        label,
        color="white",
        fontsize=10,
        weight="bold",
        bbox=dict(facecolor=color, alpha=0.85, edgecolor="none", pad=2)
    )

plt.title(f"Florence-2 Object Detection ({len(bboxes)} entities detected)", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

## Step 7: Exploring Additional Vision Tasks (`<DETAILED_CAPTION>` & `<DENSE_REGION_CAPTION>`)

In [ ]:
def run_vlm_task(task_name: str, img: Image.Image):
    inp = processor(text=task_name, images=img, return_tensors="pt")
    inp = {k: (v.to(device, dtype=dtype) if k == "pixel_values" and dtype != torch.float32 else v.to(device)) 
           for k, v in inp.items()}
    
    with torch.inference_mode():
        out_ids = model.generate(
            input_ids=inp["input_ids"],
            pixel_values=inp["pixel_values"],
            max_new_tokens=1024,
            num_beams=3
        )
    text = processor.batch_decode(out_ids, skip_special_tokens=False)[0]
    return processor.post_process_generation(text, task=task_name, image_size=(img.width, img.height))

# Detailed captioning
caption_result = run_vlm_task("<DETAILED_CAPTION>", image)
print("=== Detailed Scene Caption ===")
print(caption_result["<DETAILED_CAPTION>"])

# Dense region captioning
dense_result = run_vlm_task("<DENSE_REGION_CAPTION>", image)
print("\n=== Dense Region Caption Count ===", len(dense_result["<DENSE_REGION_CAPTION>"].get("bboxes", [])))

## Summary & Conclusion

In this project by **Johnny Wilson Dougherty**, we demonstrated:
1. Loading and optimizing open-source Vision-Language Models locally without relying on commercial APIs.
2. Managing hardware placement across GPU / Apple Silicon MPS / CPU.
3. Enforcing RGBA-safe image loading to prevent shape mismatch crashes.
4. Unified sequence-to-sequence prompting for Object Detection, Captioning, and Dense Visual Grounding.
5. Decoding and visually overlaying structured spatial coordinates.

**Author:** Johnny Wilson Dougherty  
**Project Repository:** `JohnnyWilson16/vision-ai-app`